# Setup

In [ ]:
# 1. Install dependencies
%conda install python=3.12
%pip install --quiet pydantic-ai sentence-transformers numpy mcp nest_asyncio openai
# 1b. Patch asyncio for Jupyter (REQUIRED — run this before any agent call)
import nest_asyncio
nest_asyncio.apply()


In [ ]:
# 2. Set your OpenRouter API key
# (Get one at https://openrouter.ai — gives access to 300+ models with one key.)
#
# Paste your key here, OR leave it as None to be prompted interactively,
# OR set it in your shell as the env var OPENROUTER_API_KEY.
OPENROUTER_API_KEY = ''    # e.g. 'sk-or-v1-...'

import os, getpass

if OPENROUTER_API_KEY:
    os.environ['OPENROUTER_API_KEY'] = OPENROUTER_API_KEY
elif 'OPENROUTER_API_KEY' not in os.environ:
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter API key: ')

In [25]:
import httpx
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent
from pydantic import BaseModel

MODEL = OpenAIChatModel(
    'anthropic/claude-haiku-4.5',
    provider=OpenAIProvider(
        base_url='https://openrouter.ai/api/v1',
        api_key=os.environ['OPENROUTER_API_KEY'],
        http_client=httpx.AsyncClient(timeout=60),
    ),
)

In [26]:
def pretty_print_trace(result):
    for i, msg in enumerate(result.all_messages()):
        print(f'\n[{i}] {type(msg).__name__}')
        for part in getattr(msg, "parts", []):
            kind = type(part).__name__
            snippet = repr(part)[:200]
            print(f'    └─ {kind}: {snippet}')

# Validators

In [27]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field

class PaperSummary(BaseModel):
    authors: list[str] | None = Field(description="List of Names and Surnames of the authors")
    achieved_accuracy: float | None

abstract = "In this work authored by John and Jane, we have achieved accuracy of 74%"
agent = Agent(model=MODEL, system_prompt="Extract relevant fields from the provided abstract.", output_type=PaperSummary)
result = agent.run_sync(f"Abstract to parse: {abstract}")
result.output

PaperSummary(authors=['John', 'Jane'], achieved_accuracy=74.0)

In [29]:
from pydantic import field_validator, model_validator

class ValidatedPaperSummary(BaseModel):
    authors: list[str] | None = Field(description="Name and Surname of the author, None if not available")
    achieved_accuracy: float | None

    @field_validator('authors')
    @classmethod
    def author_must_have_name_and_surname(cls, value: list[str] | None):
        if value is None:
            return value
        for author in value:
            if len(author.split()) < 2:
                raise ValueError("Author must have both name and surname")
        return value        
    
    @field_validator('achieved_accuracy')
    @classmethod
    def accuracy_must_be_between_0_and_1(cls, value: float | None):
        if value is None:
            return value
        if not 0 <= value <= 1:
            raise ValueError("Accuracy must be between 0 and 1")
        return value

agent = Agent(model=MODEL, system_prompt="Extract relevant fields from the provided abstract.", output_type=ValidatedPaperSummary)
result = agent.run_sync(f"Abstract to parse: {abstract}", retries=5)
result.output

ValidatedPaperSummary(authors=None, achieved_accuracy=0.74)

In [30]:
pretty_print_trace(result)


[0] ModelRequest
    └─ SystemPromptPart: SystemPromptPart(content='Extract relevant fields from the provided abstract.', timestamp=datetime.datetime(2026, 5, 23, 14, 50, 52, 817817, tzinfo=datetime.timezone.utc))
    └─ UserPromptPart: UserPromptPart(content='Abstract to parse: In this work authored by John and Jane, we have achieved accuracy of 74%', timestamp=datetime.datetime(2026, 5, 23, 14, 50, 52, 817825, tzinfo=datetime.timez

[1] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='final_result', args='{"authors": ["John","Jane"], "achieved_accuracy": 74}', tool_call_id='toolu_bdrk_01KUdPLkZcr19mMkpHDCy57L')

[2] ModelRequest
    └─ RetryPromptPart: RetryPromptPart(content=[{'type': 'value_error', 'loc': ('authors',), 'msg': 'Value error, Author must have both name and surname', 'input': ['John', 'Jane']}, {'type': 'value_error', 'loc': ('achieve

[3] ModelResponse
    └─ ToolCallPart: ToolCallPart(tool_name='final_result', args='{"authors": ["<UNKNOWN>","<UNKNOWN>"], "

# Validators +

In [31]:
# Validators can also immediately fix the data or transform/convert them

class ValidatedPaperSummary(BaseModel):
    achieved_accuracy: float | None
    
    @field_validator('achieved_accuracy')
    @classmethod
    def accuracy_must_be_between_0_and_1(cls, value: float | None):
        if value is None:
            return value
        if 1 < value <= 100: # <- ADDED this condition
            value = value / 100 # Assuming the value is a percentage
        if not 0 <= value <= 1:
            raise ValueError("Accuracy must be between 0 and 1")
        return value # Here we could return a transformed value

In [32]:
# Validators can also validate the output as a whole
from typing_extensions import Self
from pydantic import model_validator

class SummationEquation(BaseModel):
    a: int
    b: int
    result: int

    @model_validator(mode='after')
    def check_sum(self) -> Self:
        if self.a + self.b != self.result:
            raise ValueError('a + b does not equal result')
        return self

# Exercise

In [ ]:
#TODO create an agent that will find the best available method for imagenet-1k and cifar-100

ABSTRACTS = [
    {
        "title": "ConvNeXt-V2 adapters improve visual classification under limited fine-tuning",
        "year": 2024,
        "abstract": (
            "Parameter-efficient fine-tuning has become increasingly important for "
            "adapting large vision models to new classification tasks. We introduce "
            "a lightweight adapter module for ConvNeXt-V2 and evaluate it on the "
            "standard ImageNet-1K, CIFAR-100, and CIFAR-10 benchmarks. The model "
            "achieved 0.84 accuracy on ImageNet-1K, 0.91 accuracy on CIFAR-100, "
            "and 0.98 accuracy on CIFAR-10. In binary out-of-distribution detection "
            "experiments derived from CIFAR-10-C, the same model reached 0.93 AUROC. "
            "These results suggest that adapter-based fine-tuning can preserve "
            "strong general-purpose visual representations while reducing the number "
            "of trainable parameters."
        ),
    },
    {
        "title": "Masked autoencoder pretraining improves robustness in image classification",
        "year": 2024,
        "abstract": (
            "We studied whether masked autoencoder pretraining improves downstream "
            "classification robustness across widely used computer-vision benchmarks. "
            "A ViT-B/16 model was pretrained with random patch masking and then "
            "fine-tuned on ImageNet-1K, CIFAR-100, and CIFAR-10. The model obtained "
            "83% accuracy on ImageNet-1K, 89% accuracy on CIFAR-100, and 97% "
            "accuracy on CIFAR-10. On corrupted-image variants from ImageNet-C and "
            "CIFAR-10-C, the model achieved 86% AUROC for distinguishing clean from "
            "corrupted samples. The findings indicate that self-supervised "
            "pretraining improves robustness without sacrificing standard "
            "classification performance."
        ),
    },
    {
        "title": "A compact ResNet with knowledge distillation narrows the accuracy gap",
        "year": 2025,
        "abstract": (
            "Small convolutional networks remain attractive for deployment on "
            "resource-constrained devices, but they often underperform larger "
            "architectures. We trained a compact ResNet using knowledge distillation "
            "from a high-capacity vision transformer and benchmarked it on "
            "ImageNet-1K, CIFAR-100, and CIFAR-10. The student model achieved "
            "0.76 accuracy on ImageNet-1K, 0.84 accuracy on CIFAR-100, and 0.95 "
            "accuracy on CIFAR-10. In an auxiliary one-vs-rest evaluation on "
            "CIFAR-100 superclass labels, the model reached 0.88 AUROC. These "
            "results show that distillation can substantially improve small-model "
            "performance while retaining efficient inference."
        ),
    },
    {
        "title": "Data augmentation policies improve cross-dataset generalisation",
        "year": 2025,
        "abstract": (
            "Automated data augmentation can improve generalisation, but its effects "
            "vary across datasets and architectures. We evaluated a learned "
            "augmentation policy using a DeiT-S classifier trained and tested on "
            "ImageNet-1K, CIFAR-100, and CIFAR-10. The model reached 82% accuracy "
            "on ImageNet-1K, 88% accuracy on CIFAR-100, and 96% accuracy on "
            "CIFAR-10. When evaluated for confidence-based failure detection, it "
            "reported 85% AUROC on ImageNet-1K validation predictions and 90% "
            "AUROC on CIFAR-100 validation predictions. The augmentation policy "
            "was most beneficial for medium-sized training regimes and improved "
            "calibration as well as top-1 classification accuracy."
        ),
    },
    {
        "title": "Graph neural networks for molecular property prediction using MoleculeNet",
        "year": 2025,
        "abstract": (
            "We developed a graph neural network for molecular property prediction "
            "using atom-level message passing and bond-aware attention. Unlike the "
            "image-classification studies, this work did not evaluate on ImageNet-1K, "
            "CIFAR-100, or CIFAR-10 because its task involved molecular graphs rather "
            "than natural images. Instead, the model was evaluated on MoleculeNet "
            "benchmarks including Tox21, ClinTox, and HIV. It achieved 78% accuracy "
            "on Tox21, 74% accuracy on ClinTox, and 81% accuracy on HIV. The model "
            "also reached 86% AUROC on Tox21 and 91% AUROC on HIV, indicating strong "
            "performance on binary molecular classification tasks."
        ),
    },
]
